In [ ]:
import pickle
import glob
import random
import h5py
import numpy as np

from scipy.special import eval_hermite

import scfitpy
from scfitpy.image_processing import (
    apply_sato_filter,
    find_contours,
    assign_contours,
    determine_regions,
    determine_peak_positions,
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# Configure matplotlib
plt.rcParams["font.size"] = 25
plt.rcParams["mathtext.fontset"] = "stix"

In [ ]:
# Load spectrum
_dat = np.load("./assets/cwSpec_currentSweep_202407011423.npz")
current = _dat["current"]  # mA
freq = _dat["freq"]  # GHz

mag1 = _dat["mag1"].T  # 2d-spectrum date, dims=[freq, current]

print(f"{mag1.shape=} {current.shape=} {freq.shape=}")

In [ ]:
# Check the spectrum
fig, ax = plt.subplots(figsize=(8, 6))

X, Y = np.meshgrid(current, freq)
mappable = ax.pcolor(X, Y, mag1, cmap="summer")
cbar = plt.colorbar(mappable, ax=ax, format="%.2f")

plt.xlabel(r"Current [mA]")
plt.ylabel(r"$\omega_p\ $[GHz]")
plt.show()

In [ ]:
mag1_filtered = apply_sato_filter(
    mag1, 4, black_ridges=True, with_plot=True, filename_plot="./outs/sato.png"
)

In [ ]:
cont_list = find_contours(
    mag1_filtered, level=0.016, with_plot=True, filename_plot="./outs/cont.png"
)
print(f"# of contours: {len(cont_list)}")

In [ ]:
cont_dict = assign_contours(
    cont_list,
    dict(
        a=(12, 11), b=(17, 18), c=(16, 15)
    ),  # この後の論理積処理ではｍ辞書内で前にあるほど優先される
    with_plot=True,
    filename_plot="./outs/cont_assignments.png",
)

In [ ]:
region_dict = determine_regions(mag1, cont_dict, 5, with_plot=True)


In [ ]:
peak_dict = determine_peak_positions(
    mag1_filtered,
    region_dict,
    xaxis=current,
    yaxis=freq,
    with_plot=True,
)

for kw, pos in peak_dict.items():
    print(f'band:{kw} --> (xval,yval)={pos}')

In [ ]:
raise  # こっから下は朝永さんのコード, TODO: 整理する

In [ ]:
cur_p, band_min = qfitter.peakTrace(
    result, freq, current, cont_band, 0, 5, 4, black_ridges=False
)

In [ ]:
# with open('2Q_tracePT_band.pickle', mode='wb') as fo:
#     pickle.dump(band_min, fo)
# with open('2Q_tracePT_cur.pickle', mode='wb') as fo:
#     pickle.dump(cur_p, fo)

In [ ]:
f = open("2Q_tracePT_band.pickle", "rb")
band_min = pickle.load(f)
f = open("2Q_tracePT_cur.pickle", "rb")
cur_p = pickle.load(f)

#### from matplotlib.colors import LogNorm
fig, ax = plt.subplots(figsize=(8, 6))
plt.rcParams["font.size"] = 25
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"
X, Y = np.meshgrid(freq, current)
mappable = ax.pcolor(Y, X, qfitter.normalization(mag1), cmap="summer")
cbar_num_format = "%.2f"
cbar = plt.colorbar(mappable, ax=ax, format=cbar_num_format)

for i in range(len(band_min)):
    ax.plot(cur_p[i], band_min[i], "*", ms=5, label=str(i))

# ax.set_ylim([5,6])
ax.set_ylabel(r"$\omega_p$ [GHz]", fontsize=24)
ax.set_xlabel(r"Current [mA]", fontsize=24)
fig.tight_layout()
plt.savefig("TracedFig.png", bbox_inches="tight", pad_inches=0.5)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm

# 3Dプロットの準備
# fig = plt.figure(figsize=(10, 7))
fig, ax = plt.subplots(figsize=(2.5, 4))
clist_p = []

i = 0

n_values = [115]  # nの値
for n in n_values:
    i = i + 1
    c = cur_p[1][n]
    f = band_min[1][n]
    p = np.argmin(abs(c - current))
    v = qfitter.normalization(mag1)[p][np.argmin(abs(freq - f))]
    n_array = np.full_like(freq, c)

    plt.plot(
        freq,
        qfitter.normalization(mag1)[p] + 4 * 0.6,
        "-",
        lw=0.5,
        color=cm.hsv(i / 10),
        label=str(c),
    )
    plt.plot(f, v + 4 * 0.6, "*", color="black")
    clist_p.append(c)

n_values = [3, 8, 25]  # nの値

for n in n_values:
    i = i + 1
    c = cur_p[2][n]
    n2 = np.argmin(abs(c - cur_p[1]))
    f = band_min[2][n]
    p = np.argmin(abs(c - current))
    v = qfitter.normalization(mag1)[p][np.argmin(abs(freq - f))]
    n_array = np.full_like(freq, c)

    plt.plot(
        freq,
        qfitter.normalization(mag1)[p] + (i - 1) * 0.6,
        "-",
        lw=0.5,
        color=cm.hsv(i / 10),
        label=str(c),
    )
    plt.plot(f, v + (i - 1) * 0.6, "*", color="black")

    c2 = cur_p[1][n2]
    f2 = band_min[1][n2]
    p2 = np.argmin(abs(c2 - current))
    v2 = qfitter.normalization(mag1)[p2][np.argmin(abs(freq - f2))]
    n_array2 = np.full_like(freq, c2)

    # plt.plot(freq, qfitter.standardization(mag1)[p2], '-',lw=0.5,color=cm.hsv(i/10))
    plt.plot(f2, v2 + (i - 1) * 0.6, "*", color="black")
    clist_p.append(c)

# ラベルと凡例を設定
ax.set_ylabel("Amplitude", labelpad=15)
ax.set_xlabel(r"$\omega_p$ [GHz]", labelpad=15)
# ax.legend()

# ラベル位置をさらに改善
ax.xaxis.label.set_size(20)
ax.yaxis.label.set_size(20)
ax.tick_params(axis="both", which="major", labelsize=20)  # 軸目盛のサイズ調整
ax.tick_params(labelbottom=True, labelleft=False, labelright=False, labeltop=False)
ax.tick_params(bottom=True, left=False, right=False, top=False)

# fig.tight_layout(pad=0.1)  # 全体の余白を調整
plt.subplots_adjust(left=0, right=2, top=2, bottom=1)  # さらに微調整
# プロットを表示
plt.savefig("PT2D.png", bbox_inches="tight", pad_inches=0.5)
plt.show()
clist_p

In [ ]:
#### from matplotlib.colors import LogNorm
fig, ax = plt.subplots(figsize=(8, 6))
plt.rcParams["font.size"] = 25
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"
X, Y = np.meshgrid(freq, current)
mappable = ax.pcolor(Y, X, qfitter.normalization(mag1), cmap="summer")
cbar_num_format = "%.2f"
cbar = plt.colorbar(mappable, ax=ax, format=cbar_num_format)

for i in range(len(band_min)):
    ax.plot(cur_p[i], band_min[i], "*", ms=5, label=str(i))

for c in clist_p:
    plt.vlines(c, max(freq), 5.45, color="black", lw=1)

# ax.set_ylim([5,6])
ax.set_ylabel(r"$\omega_p$ [GHz]", fontsize=24)
ax.set_xlabel(r"Current [mA]", fontsize=24)
fig.tight_layout()
plt.savefig("TracedFig.png", bbox_inches="tight", pad_inches=0.5)
plt.show()